In [1]:
!pip install langchain langchain-community langchain-openai

  Using cached openai-2.37.0-py3-none-any.whl.metadata (31 kB)
Using cached openai-2.37.0-py3-none-any.whl (1.3 MB)
  Attempting uninstall: openai
    Found existing installation: openai 0.28.0
    Uninstalling openai-0.28.0:
      Successfully uninstalled openai-0.28.0



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from pydantic import BaseModel, Field
from typing import List, Optional

from langchain_core.prompts import PromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import PydanticOutputParser
from langchain_community.document_loaders import PyPDFLoader

In [3]:
!pip install langchain


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
# create class for each main category
class Education(BaseModel):
    university_name: str = Field(..., description='Name of the university')
    degree: str = Field(..., description='Degree Obtained')
    gpa: Optional[float] = Field(None, ge=0, le=10.0, description='GPA (0-10 scale)')

class Experience(BaseModel):
    company_name: Optional[str] = Field(..., description='Name of the company')
    n_years: Optional[int] = Field(..., ge=0, description='Number of years of experience in the company')
    project_name: Optional[str] = Field(..., description='Name of the Main Project')
    project_description: Optional[str] = Field(..., description='Role and achievements described in the project')
    tech_stack: Optional[str] = Field(..., description='Technologies and Tools used in the project')

class Resume(BaseModel):
    name: str = Field(..., description='Full name of the candidate')
    age: Optional[int] = Field(None, ge=0, description='Age of the candidate')
    email: str = Field(..., description='Email Address')
    phone_number: str = Field(..., description='Phone Number')
    experience: Optional[List[Experience]] = Field(..., description='List of professional Experience')
    education: Optional[List[Education]] = Field(..., description='Educational Background')
    languages: Optional[str] = Field(..., description='Languages Known')

In [8]:
resume_template = """
You are an AI assistant tasked with extracting structured information from a technical resume.

Only Extract the information that's present in the Resume class.

Resume Context:
{resume_text}
"""

prompt_template = PromptTemplate(
    template = resume_template,
    input_variables=['resume_text']
)

In [9]:
# initialize the model
model = init_chat_model(model='gpt-4o-mini', model_provider='openai').with_structured_output(Resume, method="function_calling")

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [10]:
# load the document
file_path = 'data/johndoe_resume.pdf'
loader = PyPDFLoader(file_path)

docs = loader.load()

resume_text = "\n".join([doc.page_content for doc in docs])
len(resume_text)

ValueError: File path data/johndoe_resume.pdf is not a valid file or url

In [11]:
# call the application
prompt = prompt_template.invoke({'resume_text': resume_text})
response = model.invoke(prompt)
response.model_dump()

NameError: name 'resume_text' is not defined